## Imports

In [1]:
import os
import json
import random
import shutil
import tempfile

import torch

from huggingface_hub import login, HfApi, hf_hub_download
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
!pip install -q tokenizers 
!pip install ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 3.1 MB/s eta 0:00:00


In [3]:
!pip install -q bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.4 MB/s eta 0:00:00


In [4]:
import re
import ijson
from collections import Counter

from tokenizers import ByteLevelBPETokenizer

## Configuration


In [5]:
# =========================
# PROJECT CONFIGURATION
# =========================

# Hugging Face
HF_USERNAME = "A-240031"
HF_REPO_NAME = "legal-lm-phase1"
HF_MODEL_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"
HF_DATASET_REPO_ID = "Exploration-Lab/CS-781-Capstone"
# Dataset files
DATASET_FILES = {
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

# Checkpointing
CHECKPOINT_INTERVAL_TOKENS = 1_000_000

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


#TOKENIZER CONFIGURATION
VOCAB_SIZE = 32_768
MIN_FREQUENCY = 5
CUSTOM_TOKEN_MIN_FREQUENCY = 100

SPECIAL_TOKENS = [
    "<|pad|>",
    "<|unk|>",
    "<|bos|>",
    "<|eos|>",
]

ARCH_TRAIN_TOKENS = 50_000_000
VAL_EXAMPLES = 500
VAL_PROMPT_WORDS = 512
VAL_TARGET_WORDS = 128
VAL_DOC_FRACTION = 0.01
SEED = 42

ARCH_TRAIN_PATH = "/kaggle/working/architecture_data/arch_train.bin"
ARCH_VAL_PATH = "/kaggle/working/architecture_data/arch_val.json"


# =========================
# TRAINING CONFIGURATION
# =========================


In [6]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Checkpoint related functions

In [7]:
def save_checkpoint(tag, model, optimizer, scheduler=None, scaler=None,
                    tokens_seen=0, global_step=0, epoch=0, extra_state=None):

    with tempfile.TemporaryDirectory() as tmp_dir:

        torch.save(model.state_dict(), os.path.join(tmp_dir, "model.pt"))
        torch.save(optimizer.state_dict(), os.path.join(tmp_dir, "optimizer.pt"))

        if scheduler is not None:
            torch.save(scheduler.state_dict(), os.path.join(tmp_dir, "scheduler.pt"))

        if scaler is not None:
            torch.save(scaler.state_dict(), os.path.join(tmp_dir, "scaler.pt"))

        rng_state = {"python": random.getstate(), "torch": torch.get_rng_state()}

        if torch.cuda.is_available():
            rng_state["cuda"] = torch.cuda.get_rng_state_all()

        torch.save(rng_state, os.path.join(tmp_dir, "rng_state.pt"))

        state = {
            "tag": tag,
            "tokens_seen": tokens_seen,
            "global_step": global_step,
            "epoch": epoch,
            "extra_state": extra_state,
        }

        with open(os.path.join(tmp_dir, "training_state.json"), "w") as f:
            json.dump(state, f, indent=2)

        HfApi().upload_folder(
            repo_id=HF_MODEL_REPO_ID,
            repo_type="model",
            folder_path=tmp_dir,
            path_in_repo=f"checkpoints/{tag}",
            commit_message=f"Save checkpoint: {tag}",
        )

def load_checkpoint(tag, model, optimizer, scheduler=None, scaler=None):

    with tempfile.TemporaryDirectory() as tmp_dir:

        files = [
            "model.pt",
            "optimizer.pt",
            "scheduler.pt",
            "scaler.pt",
            "rng_state.pt",
            "training_state.json",
        ]

        checkpoint_dir = os.path.join(
            tmp_dir,
            "checkpoints",
            tag,
        )

        for filename in files:
            try:
                hf_hub_download(
                    repo_id=HF_MODEL_REPO_ID,
                    repo_type="model",
                    filename=f"checkpoints/{tag}/{filename}",
                    local_dir=tmp_dir,
                )
            except Exception:
                pass

        model.load_state_dict(
            torch.load(
                os.path.join(checkpoint_dir, "model.pt"),
                map_location=DEVICE,
                weights_only=True,
            )
        )

        optimizer.load_state_dict(
            torch.load(
                os.path.join(checkpoint_dir, "optimizer.pt"),
                map_location=DEVICE,
                weights_only=False,
            )
        )

        scheduler_path = os.path.join(checkpoint_dir, "scheduler.pt")

        if scheduler is not None and os.path.exists(scheduler_path):
            scheduler.load_state_dict(
                torch.load(
                    scheduler_path,
                    map_location=DEVICE,
                    weights_only=False,
                )
            )

        scaler_path = os.path.join(checkpoint_dir, "scaler.pt")

        if scaler is not None and os.path.exists(scaler_path):
            scaler.load_state_dict(
                torch.load(
                    scaler_path,
                    map_location=DEVICE,
                    weights_only=False,
                )
            )

        rng_state = torch.load(
            os.path.join(checkpoint_dir, "rng_state.pt"),
            map_location="cpu",
            weights_only=False,
        )

        random.setstate(rng_state["python"])
        torch.set_rng_state(rng_state["torch"])

        if torch.cuda.is_available() and "cuda" in rng_state:
            torch.cuda.set_rng_state_all(rng_state["cuda"])

        with open(
            os.path.join(checkpoint_dir, "training_state.json")
        ) as f:
            state = json.load(f)

    return state

## Loading training data

In [8]:
from huggingface_hub import hf_hub_download


def download_dataset(name):
    return hf_hub_download(
        repo_id=HF_DATASET_REPO_ID,
        repo_type="dataset",
        filename=DATASET_FILES[name],
    )

In [9]:
SUPREME_COURT_PATH = download_dataset("supreme_court")

Phase_1_Pretraining/Train/train_supreme_(…):   0%|          | 0.00/1.32G [00:00<?, ?B/s]

In [10]:
def corpus_iterator(path):
    with open(path, "rb") as f:
        for record in ijson.items(f, "item"):
            text = record.get("text")
            if text:
                yield text

In [11]:
tokenizer = ByteLevelBPETokenizer(add_prefix_space=False,lowercase=False)
tokenizer.train_from_iterator(corpus_iterator(SUPREME_COURT_PATH),vocab_size=VOCAB_SIZE,min_frequency=MIN_FREQUENCY,special_tokens=SPECIAL_TOKENS)

In [12]:
tokenizer

Tokenizer(vocabulary_size=32768, model=ByteLevelBPE, add_prefix_space=False, lowercase=False, dropout=None, unicode_normalizer=None, continuing_subword_prefix=None, end_of_word_suffix=None, trim_offsets=False)

In [13]:
def add_custom_legal_tokens(tokenizer, corpus_path, min_frequency=100):
    patterns = [
        re.compile(r"\b(?:Article|ARTICLE)\s+\d+[A-Za-z]*(?:\([0-9A-Za-z]+\))*"),
        re.compile(r"\b(?:Section|SECTION)\s+\d+[A-Za-z]*(?:\([0-9A-Za-z]+\))*"),
        re.compile(r"\b(?:Rule|RULE)\s+\d+[A-Za-z]*(?:\([0-9A-Za-z]+\))*"),
        re.compile(r"\b(?:Regulation|REGULATION)\s+\d+[A-Za-z]*(?:\([0-9A-Za-z]+\))*"),
        re.compile(r"\([a-z]\)"),
        re.compile(r"\([0-9]{1,2}\)"),
        re.compile(r"\((?:i{1,3}|iv|v|vi{0,3}|ix|x{1,2})\)"),
    ]

    counts = Counter()

    for text in corpus_iterator(corpus_path):
        for pattern in patterns:
            counts.update(pattern.findall(text))

    custom_tokens = []

    for token, count in counts.items():
        token_with_space = "Ġ" + token

        if count >= min_frequency and tokenizer.token_to_id(token_with_space) is None:
            custom_tokens.append(token_with_space)

    tokenizer.add_tokens(custom_tokens)

    return custom_tokens, counts

In [14]:
custom_tokens, custom_counts = add_custom_legal_tokens(tokenizer,SUPREME_COURT_PATH,CUSTOM_TOKEN_MIN_FREQUENCY)

print("Custom tokens added:", len(custom_tokens))
print("Final vocabulary size:", tokenizer.get_vocab_size())

Custom tokens added: 1119
Final vocabulary size: 33887


In [15]:
vocab = tokenizer.get_vocab()
print(list(vocab)[:100])

VOCAB_SIZE = len(vocab)

['Ġrand', 'ĠInf', 'Ġwife', 'conomic', 'Ġmixed', 'manand', 'representation', 'rary', 'Ġalienees', 'ĠMUK', 'ciplinary', 'Ġpremed', 'For', 'ĠPattan', 'ĠVill', 'ĠRule 62', 'leged', 'Revision', 'Ġannounce', 'Ġthereunder', 'ĠKurukshetra', 'PUR', 'ĠST', 'ĠWherever', 'forming', 'ĠORS', 'riding', 'Ġ458', 'ĠMaruti', 'rab', 'Ġ243', 'weigh', 'Ġsubscribing', 'Workmen', 'Ġproperly', 'Ġpoli', 'Ġdissatisfaction', 'ĠCase', 'oy', 'Ġrid', 'ĠAtul', 'Ġcorroboration', 'tle', 'mount', 'geons', 'Ġaccession', 'Ġgroundless', 'during', 'Ġshr', 'argah', 'Ġunconscion', 'Mod', 'Ġimprove', 'Ġlinking', 'Ġinsight', 'ĠTehsil', 'Ġdefendants', 'Ġtourist', 'Ġbelievable', 'Ġdespatches', 'Ġcapitation', 'ĠDistilleries', 'ĠSharma', 'Ġjustifies', 'ivision', 'acket', 'dependent', 'Ġdestitute', 'Ġmeasured', 'Ġrefere', 'Ġpayers', 'Ġassignment', 'Ġten', 'Ġhat', 'ĠRoy', 'ĠAs', 'ĠRamlal', 'Ġcitizenship', 'gence', 'Ġlocom', 'Ġaffirmation', 'ĠSection 324', 'rea', 'ĠSection 326', 'veloper', 'FR', 'ss', 'ĠKhas', 'Ġmeters', 'Ġxxx', 'ĠOre

In [16]:
text = "The Court held that Section 302 applies under Article 21 and Rule 6(2)."

encoding = tokenizer.encode(text)

print("TOKENS:")
print(encoding.tokens)

print("\nDECODED:")
print(repr(tokenizer.decode(encoding.ids)))

print("\nMATCH:")
print(tokenizer.decode(encoding.ids) == text)

TOKENS:
['The', 'ĠCourt', 'Ġheld', 'Ġthat', 'ĠSection', 'Ġ302', 'Ġapplies', 'Ġunder', 'ĠArticle', 'Ġ21', 'Ġand', 'ĠRule', 'Ġ6', '(', '2', ').']

DECODED:
'The Court held that Section 302 applies under Article 21 and Rule 6(2).'

MATCH:
True


In [17]:
from pathlib import Path
from huggingface_hub import HfApi

TOKENIZER_DIR = Path("tokenizer")
TOKENIZER_DIR.mkdir(exist_ok=True)

tokenizer.save_model(str(TOKENIZER_DIR))
tokenizer.save(str(TOKENIZER_DIR / "tokenizer.json"))

HfApi().upload_folder(
    folder_path=str(TOKENIZER_DIR),
    repo_id=HF_MODEL_REPO_ID,
    repo_type="model",
    path_in_repo="tokenizer",
)

print("Tokenizer saved to:", f"{HF_MODEL_REPO_ID}/tokenizer")

No files have been modified since last commit. Skipping to prevent empty commit.


Tokenizer saved to: A-240031/legal-lm-phase1/tokenizer


## Splitting dataset into train and val set for choosing architecture

In [18]:
import numpy as np
def load_or_create_architecture_data(path):

    try:
        hf_hub_download(
            repo_id=HF_MODEL_REPO_ID,
            repo_type="model",
            filename="architecture_data/arch_train.bin",
            local_dir="/kaggle/working",
        )

        hf_hub_download(
            repo_id=HF_MODEL_REPO_ID,
            repo_type="model",
            filename="architecture_data/arch_val.json",
            local_dir="/kaggle/working",
        )

        print("Loaded architecture data from Hugging Face.")
        return

    except Exception:
        print("Architecture data not found. Creating it...")

    docs = list(corpus_iterator(path))

    split = int(len(docs) * VAL_DOC_FRACTION)

    val_docs = docs[:split]
    train_docs = docs[split:]

    with open(ARCH_TRAIN_PATH, "wb") as f:
        for text in train_docs:
            ids = tokenizer.encode(text).ids
            ids.append(tokenizer.token_to_id("<|eos|>"))
            np.asarray(ids, dtype=np.uint16).tofile(f)

    val_examples = []

    for text in val_docs:
        words = text.split()

        if len(words) < VAL_PROMPT_WORDS + VAL_TARGET_WORDS:
            continue

        for start in range(
            0,
            len(words) - VAL_PROMPT_WORDS - VAL_TARGET_WORDS + 1,
            VAL_PROMPT_WORDS,
        ):
            val_examples.append({
                "prompt": " ".join(words[start:start + VAL_PROMPT_WORDS]),
                "target": " ".join(
                    words[
                        start + VAL_PROMPT_WORDS:
                        start + VAL_PROMPT_WORDS + VAL_TARGET_WORDS
                    ]
                ),
            })

            if len(val_examples) >= VAL_EXAMPLES:
                break

        if len(val_examples) >= VAL_EXAMPLES:
            break

    with open(ARCH_VAL_PATH, "w", encoding="utf-8") as f:
        json.dump(val_examples, f, ensure_ascii=False)

    HfApi().upload_file(
        path_or_fileobj=ARCH_TRAIN_PATH,
        path_in_repo="architecture_data/arch_train.bin",
        repo_id=HF_MODEL_REPO_ID,
        repo_type="model",
    )

    HfApi().upload_file(
        path_or_fileobj=ARCH_VAL_PATH,
        path_in_repo="architecture_data/arch_val.json",
        repo_id=HF_MODEL_REPO_ID,
        repo_type="model",
    )

    print("Architecture data created and uploaded.")

In [19]:
load_or_create_architecture_data(SUPREME_COURT_PATH)


architecture_data/arch_train.bin:   0%|          | 0.00/508M [00:00<?, ?B/s]

arch_val.json: 0.00B [00:00, ?B/s]

Loaded architecture data from Hugging Face.


In [20]:
VOCAB_SIZE = tokenizer.get_vocab_size()

D_MODEL = 768
N_LAYERS = 12
N_HEADS = 12
N_KV_HEADS = 4          # GQA: 4 KV heads, 12 query heads
D_FF = 3072
SEQ_LEN = 2048

BATCH_SIZE = 4

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
GRAD_CLIP = 5.0

ARCH_TRAIN_TOKENS = 1_000_000
CHECKPOINT_INTERVAL_TOKENS = 1_000_000

In [21]:
# =========================
# SEQUENTIAL TOKEN DATASET
# =========================
SEQ_LEN = 2048
from torch.utils.data import Dataset, DataLoader
class TokenDataset(Dataset):
    def __init__(self, path, seq_len):
        self.tokens = np.memmap(path, dtype=np.uint16, mode="r")
        self.seq_len = seq_len
        self.num_sequences = len(self.tokens) // seq_len

    def __len__(self):
        return self.num_sequences

    def __getitem__(self, idx):
        start = idx * self.seq_len
        end = start + self.seq_len

        return torch.from_numpy(
            self.tokens[start:end].astype(np.int64)
        )


train_dataset = TokenDataset(
    ARCH_TRAIN_PATH,
    SEQ_LEN,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=True,
)

TOKENS_PER_STEP = BATCH_SIZE * SEQ_LEN
TOKENS_PER_EPOCH = CHECKPOINT_INTERVAL_TOKENS
TOTAL_TOKENS = min(
    ARCH_TRAIN_TOKENS,
    len(train_dataset) * SEQ_LEN,
)



In [22]:
with open(ARCH_VAL_PATH, "r", encoding="utf-8") as f:
    val_data = json.load(f)

## Model - A1 

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# RoPE
# ============================================================

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len=2048, base=10000):
        super().__init__()

        inv_freq = 1.0 / (
            base ** (torch.arange(0, head_dim, 2).float() / head_dim)
        )

        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)

        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, q, k):
        # q, k: [B, H, T, D]
        T = q.size(2)

        cos = self.cos[:T].unsqueeze(0).unsqueeze(0)
        sin = self.sin[:T].unsqueeze(0).unsqueeze(0)

        q1, q2 = q[..., ::2], q[..., 1::2]
        k1, k2 = k[..., ::2], k[..., 1::2]

        q_rot = torch.stack(
            [q1 * cos - q2 * sin,
             q1 * sin + q2 * cos],
            dim=-1,
        ).flatten(-2)

        k_rot = torch.stack(
            [k1 * cos - k2 * sin,
             k1 * sin + k2 * cos],
            dim=-1,
        ).flatten(-2)

        return q_rot, k_rot


# ============================================================
# SwiGLU
# ============================================================

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(
            F.silu(self.gate_proj(x)) * self.up_proj(x)
        )


# ============================================================
# GQA + SDPA
# ============================================================

class GQAAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()

        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(
            d_model,
            n_heads * self.head_dim,
            bias=False,
        )

        self.k_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.v_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=False,
        )

        self.rope = RotaryEmbedding(
            self.head_dim,
            max_seq_len=2048,
        )

    def forward(self, x):

        B, T, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B, T,
            self.n_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, T,
            self.n_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, T,
            self.n_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        q, k = self.rope(q, k)

        # Repeat KV heads for GQA
        repeat_factor = self.n_heads // self.n_kv_heads

        k = k.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        v = v.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        # PyTorch SDPA can dispatch to Flash Attention
        # / memory-efficient kernels automatically.
        attn_out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=True,
        )

        attn_out = (
            attn_out
            .transpose(1, 2)
            .contiguous()
            .view(B, T, -1)
        )

        return self.out_proj(attn_out)


# ============================================================
# Decoder Block
# ============================================================

class DecoderBlock(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        n_kv_heads,
        d_ff,
    ):
        super().__init__()

        self.norm1 = nn.RMSNorm(d_model)
        self.attention = GQAAttention(
            d_model,
            n_heads,
            n_kv_heads,
        )

        self.norm2 = nn.RMSNorm(d_model)
        self.ffn = SwiGLU(
            d_model,
            d_ff,
        )

    def forward(self, x):

        x = x + self.attention(
            self.norm1(x)
        )

        x = x + self.ffn(
            self.norm2(x)
        )

        return x


# ============================================================
# Decoder-only Language Model
# ============================================================

class EfficientDecoderLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=768,
        n_heads=12,
        n_kv_heads=4,
        n_layers=12,
        d_ff=3072,
        max_seq_len=2048,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model

        self.token_embedding = nn.Embedding(                             
            vocab_size,
            d_model,
        )

        self.blocks = nn.ModuleList([
            DecoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                d_ff=d_ff,
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.RMSNorm(d_model)

        # Tied LM head
        self.lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False,
        )

        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)

        elif isinstance(module, nn.Embedding):
            nn.init.xavier_uniform_(module.weight)

        elif isinstance(module, nn.RMSNorm):
            nn.init.ones_(module.weight)

    def forward(self, input_ids):

        x = self.token_embedding(input_ids)

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)

        return self.lm_head(x)

In [24]:
d_model    = 512
n_layers   = 12
n_heads    = 16
n_kv_heads = 4        # GQA
d_ff       = 1408     # SwiGLU
seq_len    = 2048

In [25]:
model = EfficientDecoderLM(
    vocab_size=VOCAB_SIZE,
    d_model=d_model,
    n_heads=n_heads,
    n_kv_heads=n_kv_heads,
    n_layers=n_layers,
    d_ff=d_ff,
    max_seq_len=SEQ_LEN,
).to(DEVICE)

print(
    f"Parameters: "
    f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M"
)

Parameters: 51.18M


In [26]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.1,
)


In [27]:
import time
import torch
import torch.nn.functional as F


def train(model, train_loader, optimizer, train_tokens):
    model.train()

    tokens_seen = 0
    global_step = 0

    start_time = time.perf_counter()

    for input_ids in train_loader:

        if tokens_seen >= train_tokens:
            break

        input_ids = input_ids.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # -------------------------
        # Forward
        # -------------------------
        logits = model(input_ids)

        loss = F.cross_entropy(
            logits[:, :-1].reshape(-1, logits.size(-1)),
            input_ids[:, 1:].reshape(-1),
        )

        # -------------------------
        # Backward
        # -------------------------
        loss.backward()

        # -------------------------
        # Gradient clipping
        # -------------------------
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP,
        )

        # -------------------------
        # Optimizer
        # -------------------------
        optimizer.step()

        # -------------------------
        # Progress
        # -------------------------
        batch_tokens = input_ids.numel()
        tokens_seen += batch_tokens
        global_step += 1

        torch.cuda.synchronize()

        elapsed = time.perf_counter() - start_time
        tokens_per_sec = tokens_seen / elapsed

        print(
            f"Step {global_step:5d} | "
            f"Tokens {tokens_seen:,}/{train_tokens:,} | "
            f"Loss {loss.item():.4f} | "
            f"Tokens/s {tokens_per_sec:,.0f}"
        )

    print("\nTraining complete.")
    print(f"Steps:       {global_step}")
    print(f"Tokens:      {tokens_seen:,}")
    print(f"Time:        {elapsed:.2f}s")
    print(f"Tokens/sec:  {tokens_seen / elapsed:,.0f}")